# 第 4 章 第一个智能体循环

## 从一问一答到一直聊下去

到目前为止，学习助手已经有了身份和风格，但每次只能回答一个问题：运行一次代码，问一句，答一句，就结束了。

小明想要的，是像聊天软件那样，可以一直聊下去：问完一个问题，接着问下一个，直到他说“再见”。

要做到这一点，只需要一个**循环**。

## 智能体循环

把“一直聊下去”拆开来看，其实就是不停地重复三件事：

```
        ┌──────────────────────────────┐
        ▼                              │
  1. 读取：小明输入一个问题               │
        │                              │
        ▼                              │
  2. 思考：把问题交给大模型               │
        │                              │
        ▼                              │
  3. 回应：把大模型的回答显示出来  ────────┘
```

这个循环看起来很简单，却是所有智能体的骨架。这本书后面的内容，都是往这个循环里加东西：

- 第 5 章，加上**记忆**：思考之前，先把之前聊过的内容也带上。
- 第 9 章，加上**工具**：大模型思考以后，可能不直接回答，而是先去查日期、查资料，再接着思考。
- 第 19 章，加上**计划**：遇到复杂的任务，先列计划，再一步步执行。

不管智能体变得多复杂，最外面都是这样一个循环。

## 前面几章的代码

下面是前 3 章写好的代码：消息类和两个 Provider。这一章没有改动它们，直接运行即可。

In [ ]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## Agent 类

现在来写这一章的主角：`Agent` 类。

- `__init__` 保存两样东西：用哪个大模型（`provider`），以及学习助手的系统消息（`system`）。
- `chat` 就是上面说的循环，用 `while True` 一直转下去：
  1. **读取**：用 `input` 读取用户输入的一行文字。
  2. 如果输入的是 `/quit`，就用 `break` 跳出循环，结束对话；如果什么都没输入，就用 `continue` 跳过这一轮。
  3. **思考**：把系统消息和用户的问题交给 `provider.call`。
  4. **回应**：把回答打印出来，然后回到第 1 步。

`chat` 还有一个参数 `questions`。在 Jupyter 里演示时，每次都手动输入问题有点麻烦，所以可以提前准备好一个问题列表，让 `chat` 一条一条地读，读完了就自动结束。如果不传 `questions`，就和平时一样用键盘输入。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            response = self.provider.call(system or self.system, [UserMessage(message)])
            print(f"AI: {response.summary()}")

## 和学习助手聊两句

我们准备两个问题：先告诉它“我叫小明”，再问它“我叫什么名字”。

In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学这本书。", "我叫什么名字？"])  # no memory yet: can it answer?
# or call agent.chat() without questions to type your own

## 它为什么记不住？

运行一下就会发现，学习助手答不出小明的名字。明明上一句刚说过，为什么就忘了呢？

问题出在这一行：

```python
response = self.provider.call(system or self.system, [UserMessage(message)])
```

每一轮循环，我们只把**当前这一句话**发给了大模型。大模型的 API 是**无状态**的：每次调用都是独立的，它不会记得上一次调用发生过什么。在它看来，每一轮都是第一次见到小明。

我们在网页上用聊天机器人时，它之所以“记得”前面的内容，是因为网页每次都把整段对话一起发了过去。下一章，我们也来做这件事：给学习助手加上**记忆**。

## 命令行应用程序

我们一直在 Jupyter 里运行代码。但真正的程序，通常是在**终端**（也叫命令行）里运行的。

像 `pip install openai`、`git commit` 这样，在终端里输入命令、用文字和程序交互的程序，就叫**命令行应用程序**（Command Line Interface，CLI）。它不需要做界面，写起来简单，也很方便和其他程序配合使用，所以很受程序员欢迎。

我们的 `chat` 方法，其实已经是一个命令行程序的核心了：读取一行输入、处理、打印结果、再读下一行。这种模式有个名字，叫 **REPL**（Read-Eval-Print Loop，读取—求值—打印循环）。在终端里输入 `python` 进入的交互环境，就是一个 REPL。

现在很多 AI 编程工具也是这样工作的。比如 Anthropic 的 **Claude Code**，就是一个在终端里运行的编程智能体：你在命令行里告诉它要做什么，它读取你的代码、修改文件、运行命令，再把结果告诉你。如果拆开来看，它的最外层也是我们这一章写的循环，只是在循环里加上了记忆、工具、计划这些本领。而这些本领，正是这本书接下来要一章一章加上去的。

## 动手试试：做一个命令行版的学习助手

把这一章的代码变成一个真正的命令行程序，在终端里运行。

**练习 1：在终端里运行。** 新建一个文件 `agent.py`，把本章的代码复制进去（导入、消息类、Provider、Agent），在最后加上：

```python
if __name__ == "__main__":
    Agent(OpenAIProvider(), system_message).chat()
```

然后在终端里运行 `python agent.py`，和学习助手聊几句，输入 `/quit` 退出。

`if __name__ == "__main__":` 的意思是：只有直接运行这个文件时，才执行下面的代码；如果这个文件被别的程序 `import`，就不执行。

**练习 2：用命令行参数选择模型。** 让程序支持 `python agent.py --provider claude`，用参数决定使用 OpenAI 还是 Claude。提示：使用 Python 自带的 `argparse` 模块。

**练习 3：一次性模式。** 让程序支持 `python agent.py "什么是变量？"`：如果命令后面跟着一个问题，就只回答这一个问题，然后退出，不进入对话循环。很多命令行工具都有这样的用法，方便在脚本里调用。

**参考答案（练习 2 和 3）：**

```python
# agent.py：把本章前面的代码（导入、消息类、Provider、Agent）复制到这里，再加上下面的部分
import argparse


def main():
    parser = argparse.ArgumentParser(description="小明的学习助手")
    parser.add_argument("question", nargs="?", help="只问一个问题，回答完就退出")
    parser.add_argument("--provider", choices=["openai", "claude"], default="openai", help="使用哪家的模型")
    args = parser.parse_args()

    provider = OpenAIProvider() if args.provider == "openai" else ClaudeProvider()
    agent = Agent(provider, system_message)

    if args.question:  # 一次性模式：回答完就退出
        print(provider.call(system_message, [UserMessage(args.question)]).summary())
    else:              # 对话模式：一直聊，直到输入 /quit
        agent.chat()


if __name__ == "__main__":
    main()
```

在终端里试一试：

```bash
python agent.py                      # 对话模式
python agent.py --provider claude    # 换成 Claude
python agent.py "什么是变量？"         # 一次性模式
python agent.py --help               # argparse 自动生成的帮助信息
```

在终端里，你输入的文字本来就会显示在屏幕上，所以 `chat` 里那行 `print(f"\nYOU: {message}")` 会让问题显示两遍。把它删掉，看起来会更清爽。

## 小结

- 智能体的骨架是一个**循环**：读取输入、交给大模型思考、给出回应，然后重复。后面的章节都是往这个循环里加本领。
- `Agent` 类用 `while True` 实现这个循环，输入 `/quit` 结束对话。
- 大模型的 API 是**无状态**的，每次调用都是独立的。只发送当前的一句话，它就记不住之前聊过什么。
- 我们写的 `chat` 是一个 **REPL**，也是命令行应用程序的核心。Claude Code 这样的命令行智能体，最外层也是这样一个循环。

下一章，我们给学习助手加上记忆，让它记住小明说过的话。